# 03 - Model 1: Item-Based Collaborative Filtering
Item-Based CF finds similar movies based on user rating patterns
and recommends movies similar to what a user has already liked.

## Why Item-Based CF?
- Intuitive approach — finds movies similar to ones user liked
- Explainable — "recommended because you watched X"
- Good baseline to compare against advanced methods
- Well studied in recommendation system literature

In [ ]:
# Cell 1 - setup
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/netflix_project'

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import pickle

# install surprise
!pip install scikit-surprise

from surprise import Dataset, Reader, KNNWithMeans
from surprise.model_selection import train_test_split
from surprise import accuracy

os.makedirs(f'{BASE}/models', exist_ok=True)
os.makedirs(f'{BASE}/data/processed', exist_ok=True)

df = pd.read_csv(f'{BASE}/data/processed/ratings_sampled.csv')
movies = pd.read_csv(f'{BASE}/data/processed/movies.csv')

print("Setup complete!")
print(f"Ratings: {df.shape}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Setup complete!
Ratings: (18829034, 4)


## Sampling for Memory Efficiency
500,000 ratings sampled for model training due to RAM constraints.
This is sufficient for a robust recommendation model.

In [ ]:
# Cell 2
df_model = df.sample(n=1000000, random_state=42).reset_index(drop=True)
print(f"Reduced dataset: {df_model.shape}")
print(f"Users: {df_model['user_id'].nunique():,}")
print(f"Movies: {df_model['movie_id'].nunique():,}")

Reduced dataset: (1000000, 4)
Users: 174,422
Movies: 15,993


## Preparing Data

In [ ]:
# Cell 3 - prepare data for Surprise
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(df_model[['user_id', 'movie_id', 'rating']], reader)

trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

print("Trainset size:", trainset.n_ratings)
print("Testset size:", len(testset))

Trainset size: 800000
Testset size: 200000


## Saving Train/Test Split

In [ ]:
# Cell 3 - save testset
testset_df = pd.DataFrame(testset, columns=['user_id', 'movie_id', 'rating'])
testset_df.to_csv(f'{BASE}/data/processed/testset.csv', index=False)
print("Testset saved!")
print(testset_df.shape)

Testset saved!
(200000, 3)


## Training Item-Based CF
Cosine similarity used to find similar movies.
k=30 nearest neighbors selected for each prediction.

In [ ]:
# Cell 4 - train Item-Based CF

sim_options = {
    'name': 'cosine',
    'user_based': False
}

item_cf = KNNWithMeans(k=30, sim_options=sim_options)
item_cf.fit(trainset)
print("Training complete!")

Computing the cosine similarity matrix...
Done computing similarity matrix.
Training complete!


## Evaluation

In [ ]:
# Cell 6 - evaluate
predictions = item_cf.test(testset)
rmse = accuracy.rmse(predictions)
mae = accuracy.mae(predictions)

print(f"\nItem-Based CF Results:")
print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")

RMSE: 1.1195
MAE:  0.8630

Item-Based CF Results:
RMSE: 1.1195
MAE:  0.8630


## Saving Predictions

In [ ]:
# Cell 7 - save predictions
preds_df = pd.DataFrame([
    {'user_id': p.uid, 'movie_id': p.iid,
     'actual': p.r_ui, 'predicted': p.est}
    for p in predictions
])
preds_df.to_csv(f'{BASE}/data/processed/itemcf_predictions.csv', index=False)
print("Predictions saved!")
print(preds_df.head(10))

Predictions saved!
   user_id  movie_id  actual  predicted
0  1316350     14940     3.0   2.389175
1   199832     12155     5.0   4.970087
2   267742     15646     4.0   3.477348
3   245080     15532     3.0   2.971563
4  1648946      8317     5.0   3.913839
5   468724     12338     5.0   3.029282
6   273337       483     3.0   3.640057
7   371163     16113     3.0   3.297536
8  1228560     13811     5.0   4.166667
9  1601592      5191     4.0   3.180132


## Top 10 Recommendations

In [ ]:
# Cell 8 - Top 10 recommendations for varied user
def get_top10_recommendations(user_id, model, df, movies, n=10):
    rated_movies = df[df['user_id'] == user_id]['movie_id'].tolist()
    all_movies = df['movie_id'].unique()
    unrated = [m for m in all_movies if m not in rated_movies]

    predictions = [model.predict(user_id, movie_id, verbose=False)
                   for movie_id in unrated]

    confident_preds = [p for p in predictions
                       if not p.details.get('was_impossible', False)]

    if len(confident_preds) == 0:
        print("No confident predictions — try different user")
        return pd.DataFrame()

    confident_preds.sort(key=lambda x: x.est, reverse=True)
    top_n = confident_preds[:n]

    results = pd.DataFrame([
        {'movie_id': p.iid, 'predicted_rating': round(p.est, 2)}
        for p in top_n
    ])
    results = results.merge(movies[['movie_id', 'title']], on='movie_id')
    return results[['title', 'predicted_rating']]

# pick user with varied ratings
user_rating_std = df.groupby('user_id')['rating'].std()
varied_users = user_rating_std[user_rating_std > 1.0].index
sample_user = df[df['user_id'].isin(varied_users)]['user_id'].iloc[0]

print(f"User {sample_user} rating history:")
print(df[df['user_id'] == sample_user]['rating'].value_counts().sort_index())
print(f"\nItem-CF Top 10 for user {sample_user}:\n")
recs = get_top10_recommendations(sample_user, item_cf, df, movies)
print(recs.to_string(index=False))

User 814701 rating history:
rating
1     6
2    17
3    24
4    45
5    46
Name: count, dtype: int64

Item-CF Top 10 for user 814701:

                                                                      title  predicted_rating
                                                 Isle of Man TT 2004 Review                 5
Lord of the Rings: The Return of the King: Extended Edition: Bonus Material                 5
                                                         Nature: Antarctica                 5
                                                           Immortal Beloved                 5
                      ABC Primetime: Mel Gibson's The Passion of the Christ                 5
                                                          Barbarian Queen 2                 5
                                                                 Elfen Lied                 5
                      Record of Lodoss War: Chronicles of the Heroic Knight                 5
                   

## Limitation of Item-Based CF
Item-Based CF produces overconfident predictions (all 5.0) for some
users due to 99.4% sparsity. Most movies share very few common raters
making similarity scores unreliable. This directly motivates using
SVD in notebook 04 which handles sparsity much better.

## Saving Model

In [ ]:
# Cell 9 - save model
with open(f'{BASE}/models/item_cf_model.pkl', 'wb') as f:
    pickle.dump(item_cf, f)
print("Model saved!")

Model saved!
